# Robustness Checks for SSRN Submission

This notebook computes the four robustness analyses requested by the referee:

1. **Deflated Sharpe Ratio (DSR)** — Bailey & Lopez de Prado (2014)
2. **Hansen's Superior Predictive Ability (SPA) test** — Hansen (2005)
3. **Probability of Backtest Overfitting (PBO)** — Bailey, Borwein, Lopez de Prado, Zhu (2017)
4. **Net returns under transaction costs + leverage financing**
5. **Stationary bootstrap of the 180-day OOS Sharpe distribution** — Politis & Romano (1994)

Each section is one cell. Run all and paste the printed summary back to the assistant.

**Trial universe:** N = 120 strategies (60 model configs x {Vanilla, Kelly}).
**Best strategy:** `XGB_ds3_fsY_mse_kelly` (SR = 2.43, TR = 45.18%).


## Cell 1 — Setup and data loading

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
np.random.seed(42)

# --- Load data ----------------------------------------------------------------
RET_PATH  = "pipeline_data/backtest/all_strategies_daily_returns.csv"      # daily returns, 180 x 122
PRED_PATH = "pipeline_data/predictions/all_models_predictions_matrix.csv"    # raw predictions, 180 x 62
METRIC_PATH = "pipeline_data/backtest/strategy_metrics_comparison.csv"     # SR/MDD/TR per strategy

ret_df    = pd.read_csv(RET_PATH)
pred_df   = pd.read_csv(PRED_PATH)
metric_df = pd.read_csv(METRIC_PATH)

# --- Define trial universe ----------------------------------------------------
strategy_cols = [c for c in ret_df.columns if c not in ("date_id", "Buy_and_Hold")]
T = len(ret_df)               # 180 trading days
N_TRIALS = len(strategy_cols)  # 120

BEST = "XGB_ds3_fsY_mse_kelly"
ANNUALIZER = np.sqrt(252)

print(f"OOS days (T):        {T}")
print(f"Number of trials:    {N_TRIALS}")
print(f"Best strategy:       {BEST}")
print(f"Buy_and_Hold SR:     {ret_df['Buy_and_Hold'].mean()/ret_df['Buy_and_Hold'].std()*ANNUALIZER:.4f}")

# annualised SR per strategy
def ann_sr(r):
    r = np.asarray(r)
    mu, sd = r.mean(), r.std(ddof=1)
    return (mu / sd) * ANNUALIZER if sd > 0 else 0.0

sr_all = pd.Series({c: ann_sr(ret_df[c]) for c in strategy_cols})
print(f"\nSR distribution across {N_TRIALS} trials:")
print(sr_all.describe().round(4))


OOS days (T):        180
Number of trials:    120
Best strategy:       XGB_ds3_fsY_mse_kelly
Buy_and_Hold SR:     0.7643

SR distribution across 120 trials:
count    120.0000
mean       0.7868
std        0.5284
min       -0.6964
25%        0.6649
50%        0.7909
75%        0.9879
max        2.4333
dtype: float64


## Cell 2 — Deflated Sharpe Ratio (DSR)

Bailey & Lopez de Prado (2014), eq. (8). Adjusts the observed best SR for:
- Number of independent trials N
- Higher moments (skew, kurt) of the strategy's daily returns
- Sample length T

DSR > 0.95 (i.e., p < 0.05) is the standard threshold for statistical significance
against the no-skill null after multiple-testing correction.


In [2]:
# --- Deflated Sharpe Ratio (Bailey & Lopez de Prado 2014) ---------------------
def deflated_sharpe(observed_sr_ann, sr_distribution_ann, T,
                    skew, kurt, annualizer=np.sqrt(252)):
    """
    Returns DSR (probability that true SR > 0), z-stat, and adjusted SR0 threshold.

    Inputs:
      observed_sr_ann : the best strategy's annualised SR
      sr_distribution_ann : array of all trial SRs (annualised) used to estimate
                            the variance of the SR cross-section
      T : number of OOS observations
      skew, kurt : sample skew and EXCESS kurtosis of the best strategy's
                   daily returns
      annualizer : sqrt(252) for daily data
    """
    # Convert to non-annualised (per-period) SR for the formulas
    sr_obs   = observed_sr_ann / annualizer
    sr_trials = np.asarray(sr_distribution_ann) / annualizer

    N = len(sr_trials)
    var_trials = np.var(sr_trials, ddof=1)
    std_trials = np.sqrt(var_trials)

    # Expected maximum SR under the null of zero true SR (Bailey-Lopez de Prado 2014, eq.6)
    # E[max] = mu_sr + std_sr * ( (1 - gamma) * Phi^-1(1 - 1/N)
    #                            + gamma     * Phi^-1(1 - 1/(N*e)) )
    gamma = 0.5772156649  # Euler-Mascheroni
    e     = np.e
    sr0   = std_trials * ((1 - gamma) * stats.norm.ppf(1 - 1.0/N)
                           + gamma     * stats.norm.ppf(1 - 1.0/(N * e)))

    # DSR (eq. 8) -- excess kurtosis (kurt = gamma_2, not Pearson kurtosis)
    num = (sr_obs - sr0) * np.sqrt(T - 1)
    den = np.sqrt(1.0 - skew * sr_obs + 0.25 * kurt * sr_obs**2)
    z   = num / den
    dsr = stats.norm.cdf(z)

    return {
        "sr_obs_ann":      sr_obs * annualizer,
        "sr0_ann":          sr0 * annualizer,        # expected max under null
        "z_stat":          z,
        "DSR":             dsr,
        "p_value":         1 - dsr,
        "N_trials":        N,
        "T":               T,
        "skew":            skew,
        "excess_kurt":     kurt,
    }

# Compute moments of best strategy's daily returns
best_returns = ret_df[BEST].values
skew_best = stats.skew(best_returns, bias=False)
kurt_best = stats.kurtosis(best_returns, bias=False, fisher=True)  # excess kurt

dsr_result = deflated_sharpe(
    observed_sr_ann   = ann_sr(best_returns),
    sr_distribution_ann = sr_all.values,
    T                 = T,
    skew              = skew_best,
    kurt              = kurt_best,
)

print("=== Deflated Sharpe Ratio ===")
for k, v in dsr_result.items():
    if isinstance(v, float):
        print(f"  {k:18s} = {v:.6f}")
    else:
        print(f"  {k:18s} = {v}")
print()
print(f"Interpretation: After adjusting for {N_TRIALS} trials and the daily")
print(f"return distribution's higher moments, the probability that the true")
print(f"Sharpe is positive is {dsr_result['DSR']*100:.2f}%, p-value = {dsr_result['p_value']:.6f}.")
print(f"{'PASSES' if dsr_result['p_value']<0.05 else 'FAILS'} the 5% significance threshold.")


=== Deflated Sharpe Ratio ===
  sr_obs_ann         = 2.433282
  sr0_ann            = 1.370846
  z_stat             = 0.950717
  DSR                = 0.829126
  p_value            = 0.170874
  N_trials           = 120
  T                  = 180
  skew               = 1.019145
  excess_kurt        = 7.368567

Interpretation: After adjusting for 120 trials and the daily
return distribution's higher moments, the probability that the true
Sharpe is positive is 82.91%, p-value = 0.170874.
FAILS the 5% significance threshold.


## Cell 3 — Hansen (2005) Superior Predictive Ability (SPA) test

Tests H0: none of the N strategies has a positive expected loss-differential
against the benchmark (Buy_and_Hold). Uses stationary bootstrap to derive the
p-value of the studentised max-statistic across all strategies.

A p-value < 0.05 means we reject H0 and conclude **at least one strategy
genuinely outperforms B&H**, accounting for the data-snooping bias.


In [3]:
# --- Hansen (2005) SPA test ---------------------------------------------------
def stationary_bootstrap_indices(T, block_size, n_boot, rng):
    """Politis-Romano stationary bootstrap index generator."""
    p = 1.0 / block_size
    idx = np.empty((n_boot, T), dtype=np.int64)
    for b in range(n_boot):
        i = rng.integers(0, T)
        for t in range(T):
            idx[b, t] = i
            if rng.random() < p:
                i = rng.integers(0, T)
            else:
                i = (i + 1) % T
    return idx

def spa_test(strategy_returns, benchmark_returns, n_boot=2000,
             block_size=None, annualizer=np.sqrt(252), rng=None):
    """
    Hansen (2005) consistent SPA test.
    H0: max_k E[d_k] <= 0  where d_k = r_k - r_bench   (loss differential is -d_k)
    Larger SPA stat -> more evidence against H0 (i.e., strategy beats benchmark).
    """
    if rng is None:
        rng = np.random.default_rng(42)
    R    = np.asarray(strategy_returns)        # T x K
    bench = np.asarray(benchmark_returns).reshape(-1)
    T, K = R.shape
    if block_size is None:
        block_size = max(2, int(np.sqrt(T)))

    # d_{t,k} = excess return of strategy k over benchmark at t
    d = R - bench[:, None]                      # T x K
    d_bar = d.mean(axis=0)                      # K
    sigma_k = d.std(axis=0, ddof=1)             # K
    sigma_k = np.where(sigma_k < 1e-12, 1e-12, sigma_k)

    # Observed studentised statistic (Hansen 2005)
    T_n = np.max(np.maximum(np.sqrt(T) * d_bar / sigma_k, 0.0))

    # Recentering function g_c (consistent variant): drop strategies whose
    # mean is "too negative" -- below -sqrt((2*log log T)/T) * sigma_k
    A_n = -np.sqrt(2.0 * np.log(np.log(T)) / T) * sigma_k
    keep = d_bar >= A_n
    mu_c = np.where(keep, d_bar, 0.0)

    # Stationary bootstrap
    idx = stationary_bootstrap_indices(T, block_size, n_boot, rng)
    T_n_star = np.empty(n_boot)
    for b in range(n_boot):
        d_b = d[idx[b]] - mu_c                  # recenter
        d_bar_b = d_b.mean(axis=0)
        sigma_b = d_b.std(axis=0, ddof=1)
        sigma_b = np.where(sigma_b < 1e-12, 1e-12, sigma_b)
        T_n_star[b] = np.max(np.maximum(np.sqrt(T) * d_bar_b / sigma_b, 0.0))

    p_consistent = np.mean(T_n_star >= T_n)
    return {
        "T_n_observed": T_n,
        "p_value_consistent": p_consistent,
        "block_size": block_size,
        "n_boot": n_boot,
        "K_strategies": K,
        "T_obs": T,
    }

R_mat   = ret_df[strategy_cols].values        # T x 120
bh      = ret_df["Buy_and_Hold"].values

spa_result = spa_test(R_mat, bh, n_boot=2000,
                      rng=np.random.default_rng(42))

print("=== Hansen (2005) SPA test (consistent variant) ===")
for k, v in spa_result.items():
    if isinstance(v, float):
        print(f"  {k:25s} = {v:.6f}")
    else:
        print(f"  {k:25s} = {v}")

print()
print(f"Interpretation: SPA p-value = {spa_result['p_value_consistent']:.4f}.")
print(f"{'REJECT' if spa_result['p_value_consistent']<0.05 else 'FAIL TO REJECT'} H0 of no superior strategy at 5%.")


=== Hansen (2005) SPA test (consistent variant) ===
  T_n_observed              = 2.264563
  p_value_consistent        = 0.388500
  block_size                = 13
  n_boot                    = 2000
  K_strategies              = 120
  T_obs                     = 180

Interpretation: SPA p-value = 0.3885.
FAIL TO REJECT H0 of no superior strategy at 5%.


## Cell 4 — Probability of Backtest Overfitting (PBO)

Bailey, Borwein, Lopez de Prado & Zhu (2017). Uses **Combinatorially
Symmetric Cross-Validation (CSCV)**: split the OOS sample into S even
chunks, take all (S choose S/2) combinations of (train, test) splits, and
check how often the best strategy on "train" ranks below median on "test".

PBO is the fraction of splits where the best-in-train underperforms the
median in the held-out half. **PBO < 0.5 = backtest is informative;
PBO > 0.5 = overfit.**


In [4]:
# --- Probability of Backtest Overfitting (Bailey et al. 2017) -----------------
from itertools import combinations

def pbo_cscv(returns_matrix, S=10, annualizer=np.sqrt(252)):
    """
    Combinatorially Symmetric Cross-Validation PBO.
    returns_matrix: T x N daily returns
    S: number of chunks (must be even). Higher S = finer split.
    """
    M = np.asarray(returns_matrix)
    T, N = M.shape
    # trim T so it divides evenly into S
    T_use = (T // S) * S
    M = M[:T_use]
    chunks = np.array_split(np.arange(T_use), S)

    n_combos = 0
    n_overfit = 0
    logits = []
    half = S // 2
    for train_idx in combinations(range(S), half):
        test_idx = tuple(i for i in range(S) if i not in train_idx)
        train_rows = np.concatenate([chunks[i] for i in train_idx])
        test_rows  = np.concatenate([chunks[i] for i in test_idx])

        # Per-chunk SR using sample mean / std
        def sr_chunk(rows):
            r = M[rows]
            mu = r.mean(axis=0); sd = r.std(axis=0, ddof=1)
            sd = np.where(sd < 1e-12, 1e-12, sd)
            return (mu / sd) * annualizer

        sr_train = sr_chunk(train_rows)
        sr_test  = sr_chunk(test_rows)

        n_star = int(np.argmax(sr_train))                # best in train
        # rank of that strategy in the test set (1 = worst, N = best)
        order = np.argsort(sr_test)
        rank  = int(np.where(order == n_star)[0][0]) + 1
        omega = rank / (N + 1)                            # in (0, 1)
        # logit of relative rank; <0 means below median on test
        logit = np.log(omega / (1 - omega))
        logits.append(logit)

        n_combos += 1
        if logit < 0:
            n_overfit += 1

    pbo = n_overfit / n_combos
    return {
        "PBO": pbo,
        "n_combinations": n_combos,
        "median_logit": float(np.median(logits)),
        "S_chunks": S,
        "N_strategies": N,
    }

pbo_result = pbo_cscv(R_mat, S=10)
print("=== Probability of Backtest Overfitting (CSCV) ===")
for k, v in pbo_result.items():
    if isinstance(v, float):
        print(f"  {k:18s} = {v:.6f}")
    else:
        print(f"  {k:18s} = {v}")

print()
print(f"Interpretation: PBO = {pbo_result['PBO']*100:.2f}%.")
print(f"{'Backtest informative (PBO<50%).' if pbo_result['PBO']<0.5 else 'Evidence of overfitting (PBO>=50%).'}")


=== Probability of Backtest Overfitting (CSCV) ===
  PBO                = 0.607143
  n_combinations     = 252
  median_logit       = -0.980829
  S_chunks           = 10
  N_strategies       = 120

Interpretation: PBO = 60.71%.
Evidence of overfitting (PBO>=50%).


## Cell 5 — Net returns under transaction costs + leverage financing

Assumptions (conservative, citable):
- **Round-trip transaction cost**: 10 bps per unit position change.
  Costs applied as `c * |f_t - f_{t-1}|` where `f_t in [0,2]` is the
  daily allocation.
- **Leverage financing cost**: 2% annualised on the leveraged portion
  `max(f_t - 1, 0)`, applied daily as `(0.02/252) * max(f_t-1, 0)`.

**Position source:** if `all_strategies_daily_positions.csv` is present (Pipeline 03
exports raw `f_t` series directly), we use those values. Otherwise we fall back to
reconstructing `f_t` from `strategy_return_t / market_return_t` with a small-denominator
safeguard — the recovery is exact whenever the market return is well away from zero,
but introduces small errors on flat-market days.


In [9]:
# --- Net returns under costs + financing --------------------------------------
import os

COST_BPS = 10.0       # round-trip transaction cost (basis points)
LEV_RATE = 0.02       # 2% annualised on leveraged portion

c = COST_BPS / 10000.0
daily_fin = LEV_RATE / 252.0

# --- Position series: prefer raw positions exported by Pipeline 03 -----------
POS_PATH = "pipeline_data/backtest/all_strategies_daily_positions.csv"
USE_RAW_POSITIONS = os.path.exists(POS_PATH)

if USE_RAW_POSITIONS:
    pos_df = pd.read_csv(POS_PATH)
    print(f"Using RAW positions from {POS_PATH} (preferred).")
else:
    print("Raw positions file not found; falling back to f_t recovery from returns.")

market = ret_df["Buy_and_Hold"].values
eps = 1e-8

def recover_positions(strat_returns, market):
    f = np.zeros_like(strat_returns)
    last_valid = 0.0
    for t in range(len(strat_returns)):
        if abs(market[t]) > eps:
            ft = strat_returns[t] / market[t]
            ft = np.clip(ft, 0.0, 2.0)
            last_valid = ft
        else:
            ft = last_valid
        f[t] = ft
    return f

def get_positions(col, gross_returns):
    if USE_RAW_POSITIONS and col in pos_df.columns:
        return pos_df[col].values.astype(float)
    return recover_positions(gross_returns, market)

results = []
for col in strategy_cols:
    gross = ret_df[col].values.astype(float)
    f_t = get_positions(col, gross)
    df_t = np.empty_like(f_t)
    df_t[0]  = abs(f_t[0])
    df_t[1:] = np.abs(np.diff(f_t))
    tc_t = c * df_t
    fin_t = daily_fin * np.maximum(f_t - 1.0, 0.0)
    net = gross - tc_t - fin_t

    sr_gross = ann_sr(gross)
    sr_net   = ann_sr(net)
    tr_gross = np.prod(1 + gross) - 1
    tr_net   = np.prod(1 + net) - 1
    cum     = np.cumprod(1 + net) - 1
    mdd     = float(np.min(cum - np.maximum.accumulate(cum)))
    results.append((col, sr_gross, sr_net, tr_gross, tr_net, mdd,
                    f_t.mean(), df_t.mean()))

net_df = pd.DataFrame(results, columns=[
    "Strategy", "SR_gross", "SR_net", "TR_gross", "TR_net", "MDD_net",
    "avg_position", "avg_daily_turnover",
]).sort_values("SR_net", ascending=False).reset_index(drop=True)

print(f"\n=== Net returns (round-trip = {COST_BPS:.0f} bps, financing = {LEV_RATE*100:.0f}% on leverage) ===")
print(f"\nPositions source: {'raw (Pipeline 03 output)' if USE_RAW_POSITIONS else 'recovered from returns'}")
print(f"\nBest gross ({BEST}):")
print(net_df[net_df.Strategy == BEST].to_string(index=False))
print(f"\nTop 5 by SR_net:")
print(net_df.head(5).to_string(index=False))
print(f"\nFull table saved to net_returns_table.csv")
net_df.to_csv("pipeline_data/backtest/net_returns_table.csv", index=False)

print(f"\n--- Sensitivity (best strategy SR_net at different cost levels) ---")
best_gross_ret = ret_df[BEST].values.astype(float)
best_f = get_positions(BEST, best_gross_ret)
best_df_pos = np.r_[abs(best_f[0]), np.abs(np.diff(best_f))]
best_fin = daily_fin * np.maximum(best_f - 1.0, 0.0)
print(f"  avg position = {best_f.mean():.3f}, avg daily turnover = {best_df_pos.mean():.4f}")
for cost_bps in [0, 5, 10, 20, 50]:
    cc = cost_bps / 10000.0
    net_b = best_gross_ret - cc * best_df_pos - best_fin
    print(f"  cost = {cost_bps:3d} bps -> SR_net = {ann_sr(net_b):.4f}, "
          f"TR_net = {np.prod(1+net_b)-1:.4f}")

# Save best-strategy net series for Cell 6 bootstrap
best_net_series = best_gross_ret - c * best_df_pos - best_fin


Using RAW positions from pipeline_data/backtest/all_strategies_daily_positions.csv (preferred).

=== Net returns (round-trip = 10 bps, financing = 2% on leverage) ===

Positions source: raw (Pipeline 03 output)

Best gross (XGB_ds3_fsY_mse_kelly):
             Strategy  SR_gross   SR_net  TR_gross   TR_net   MDD_net  avg_position  avg_daily_turnover
XGB_ds3_fsY_mse_kelly  2.433282 1.251536  0.451798 0.200575 -0.134032      1.063922            1.016118

Top 5 by SR_net:
               Strategy  SR_gross   SR_net  TR_gross   TR_net   MDD_net  avg_position  avg_daily_turnover
  XGB_ds4_fsY_mse_kelly  2.377622 1.261633  0.375247 0.175160 -0.128589      0.829711            0.844072
  XGB_ds3_fsY_mse_kelly  2.433282 1.251536  0.451798 0.200575 -0.134032      1.063922            1.016118
XGB_ds1_fsN_mse_vanilla  1.262031 1.120030  0.146823 0.127731 -0.112041      0.998433            0.090990
REG_ds1_fsY_huber_kelly  1.490681 1.110991  0.306579 0.212474 -0.190634      1.354849            0.362

## Cell 6 — Stationary bootstrap of the 180-day Sharpe distribution

Politis & Romano (1994) stationary bootstrap on the best strategy's
daily returns. Block length = sqrt(180) ~= 13 days. 1,000 resamples.

Reports the 5%/50%/95% percentiles of the bootstrap SR distribution,
plus the proportion of bootstrap samples with SR > 1.0 and SR > 0.

This responds to referee comment #09: instead of a single point estimate,
we show the distribution of SR under resampled paths preserving the
serial dependence structure of the realised OOS window.


In [10]:
# --- Stationary bootstrap of OOS Sharpe ---------------------------------------
def stationary_bootstrap(x, block_size, n_boot, rng):
    p = 1.0 / block_size
    T = len(x)
    out = np.empty((n_boot, T))
    for b in range(n_boot):
        i = rng.integers(0, T)
        for t in range(T):
            out[b, t] = x[i]
            if rng.random() < p:
                i = rng.integers(0, T)
            else:
                i = (i + 1) % T
    return out

rng = np.random.default_rng(2026)
block_size = max(2, int(np.sqrt(T)))   # 13
n_boot = 1000

# Best strategy (gross)
best_gross = ret_df[BEST].values
samples = stationary_bootstrap(best_gross, block_size, n_boot, rng)
sr_boot = np.array([ann_sr(s) for s in samples])
tr_boot = np.array([np.prod(1+s) - 1 for s in samples])

# Best strategy (net) -- reuse best_net_series from Cell 5
samples_net = stationary_bootstrap(best_net_series, block_size, n_boot, rng)
sr_boot_net = np.array([ann_sr(s) for s in samples_net])

# Buy-and-hold
samples_bh = stationary_bootstrap(market, block_size, n_boot, rng)
sr_boot_bh = np.array([ann_sr(s) for s in samples_bh])

def summarise(name, arr):
    p5, p50, p95 = np.percentile(arr, [5, 50, 95])
    return {
        "name": name,
        "mean": float(np.mean(arr)),
        "p5":   float(p5),
        "p50":  float(p50),
        "p95":  float(p95),
        "pr_sr_gt_0":   float(np.mean(arr > 0)),
        "pr_sr_gt_1":   float(np.mean(arr > 1)),
    }

print("=== Stationary bootstrap of 180-day Sharpe distribution ===")
print(f"Block size: {block_size} days, n_boot: {n_boot}\n")

for s in [summarise("Best (gross)",      sr_boot),
          summarise("Best (net 10bps)",  sr_boot_net),
          summarise("Buy_and_Hold",      sr_boot_bh)]:
    print(f"  {s['name']:22s} | mean={s['mean']:6.3f}  "
          f"5%={s['p5']:6.3f}  50%={s['p50']:6.3f}  95%={s['p95']:6.3f}  "
          f"P(SR>0)={s['pr_sr_gt_0']:.3f}  P(SR>1)={s['pr_sr_gt_1']:.3f}")

print("\n--- Total return percentiles for best strategy (gross) ---")
p5, p50, p95 = np.percentile(tr_boot, [5, 50, 95])
print(f"  5%={p5:.4f}, 50%={p50:.4f}, 95%={p95:.4f}")


=== Stationary bootstrap of 180-day Sharpe distribution ===
Block size: 13 days, n_boot: 1000

  Best (gross)           | mean= 2.354  5%= 0.855  50%= 2.398  95%= 3.710  P(SR>0)=0.995  P(SR>1)=0.935
  Best (net 10bps)       | mean= 1.195  5%=-0.473  50%= 1.234  95%= 2.772  P(SR>0)=0.883  P(SR>1)=0.597
  Buy_and_Hold           | mean= 0.900  5%=-0.744  50%= 0.785  95%= 2.781  P(SR>0)=0.803  P(SR>1)=0.421

--- Total return percentiles for best strategy (gross) ---
  5%=0.1120, 50%=0.4271, 95%=0.9067


## Cell 7 — Summary block to paste back

Copy the printed block below and paste it into the chat. The assistant will
use these numbers to fill in the LaTeX.


In [13]:
print("="*70)
print()
print(f"[DSR]  Best SR (ann) = {dsr_result['sr_obs_ann']:.4f}")
print(f"[DSR]  Expected max SR under null = {dsr_result['sr0_ann']:.4f}")
print(f"[DSR]  z-stat = {dsr_result['z_stat']:.4f}")
print(f"[DSR]  DSR (prob true SR > 0) = {dsr_result['DSR']:.6f}")
print(f"[DSR]  p-value = {dsr_result['p_value']:.6f}")
print(f"[DSR]  Skew = {skew_best:.4f}, Excess kurt = {kurt_best:.4f}")
print(f"[DSR]  N trials = {dsr_result['N_trials']}, T = {dsr_result['T']}")
print()
print(f"[SPA]  Test stat T_n = {spa_result['T_n_observed']:.4f}")
print(f"[SPA]  p-value (consistent) = {spa_result['p_value_consistent']:.4f}")
print(f"[SPA]  Block size = {spa_result['block_size']}, n_boot = {spa_result['n_boot']}")
print()
print(f"[PBO]  PBO = {pbo_result['PBO']:.4f}")
print(f"[PBO]  Median logit = {pbo_result['median_logit']:.4f}")
print(f"[PBO]  S = {pbo_result['S_chunks']}, N strategies = {pbo_result['N_strategies']}")
print()
best_row = net_df[net_df.Strategy == BEST].iloc[0]
print(f"[NET]  Best strategy: {BEST}")
print(f"[NET]  SR gross = {best_row.SR_gross:.4f}, SR net = {best_row.SR_net:.4f}")
print(f"[NET]  TR gross = {best_row.TR_gross:.4f}, TR net = {best_row.TR_net:.4f}")
print(f"[NET]  MDD net = {best_row.MDD_net:.4f}")
print(f"[NET]  Avg position = {best_row.avg_position:.3f}, "
      f"avg daily turnover = {best_row.avg_daily_turnover:.4f}")
print()
sb = summarise("best_gross", sr_boot)
sbn = summarise("best_net",  sr_boot_net)
sbh = summarise("bh",        sr_boot_bh)
print(f"[BOOT] Best gross  SR p5/p50/p95 = {sb['p5']:.3f}/{sb['p50']:.3f}/{sb['p95']:.3f}, "
      f"P(SR>0)={sb['pr_sr_gt_0']:.3f}, P(SR>1)={sb['pr_sr_gt_1']:.3f}")
print(f"[BOOT] Best net    SR p5/p50/p95 = {sbn['p5']:.3f}/{sbn['p50']:.3f}/{sbn['p95']:.3f}, "
      f"P(SR>0)={sbn['pr_sr_gt_0']:.3f}, P(SR>1)={sbn['pr_sr_gt_1']:.3f}")
print(f"[BOOT] B&H         SR p5/p50/p95 = {sbh['p5']:.3f}/{sbh['p50']:.3f}/{sbh['p95']:.3f}")
print(f"[BOOT] Block size = {block_size}, n_boot = {n_boot}")
print("="*70)



[DSR]  Best SR (ann) = 2.4333
[DSR]  Expected max SR under null = 1.3708
[DSR]  z-stat = 0.9507
[DSR]  DSR (prob true SR > 0) = 0.829126
[DSR]  p-value = 0.170874
[DSR]  Skew = 1.0191, Excess kurt = 7.3686
[DSR]  N trials = 120, T = 180

[SPA]  Test stat T_n = 2.2646
[SPA]  p-value (consistent) = 0.3885
[SPA]  Block size = 13, n_boot = 2000

[PBO]  PBO = 0.6071
[PBO]  Median logit = -0.9808
[PBO]  S = 10, N strategies = 120

[NET]  Best strategy: XGB_ds3_fsY_mse_kelly
[NET]  SR gross = 2.4333, SR net = 1.2515
[NET]  TR gross = 0.4518, TR net = 0.2006
[NET]  MDD net = -0.1340
[NET]  Avg position = 1.064, avg daily turnover = 1.0161

[BOOT] Best gross  SR p5/p50/p95 = 0.855/2.398/3.710, P(SR>0)=0.995, P(SR>1)=0.935
[BOOT] Best net    SR p5/p50/p95 = -0.473/1.234/2.772, P(SR>0)=0.883, P(SR>1)=0.597
[BOOT] B&H         SR p5/p50/p95 = -0.744/0.785/2.781
[BOOT] Block size = 13, n_boot = 1000
